In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from scipy.stats import wilcoxon
from scipy.stats import friedmanchisquare
from statsmodels.stats.multitest import multipletests


# Load CSV file
sheet_id = "1tEf8J1kbGZEFTOkoYqIjsdE3PkqBA9bX4MdNfMoLhjQ"
# tab_gid = "908803302"
tab_gid = "1990888583" 
full_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={tab_gid}"

In [2]:
def check_significance(df, col1, col2):
    stat, p_value = wilcoxon(df[col1], df[col2], alternative='greater')
    if p_value < 0.05:
        print(f"({col1} vs {col2}): {stat}, p-value: {p_value} \033[92mThe difference is statistically significant.\033[0m")
    else:
        print(f"({col1} vs {col2}): {stat}, p-value: {p_value} \033[91mThe difference is NOT statistically significant.\033[0m")

## RQ1

In [3]:
limit = 15 #15 39
df0 = pd.read_csv(full_url, skiprows=2, usecols=["Model", "Compression", "BLEU"]).iloc[:limit, :]
df1 = df0.pivot(index="Model", columns="Compression", values="BLEU")
df = df1.reset_index()
print(df.shape)
check_significance(df, "Original", "8-bit")
check_significance(df, "Original", "4-bit")

(5, 4)
(Original vs 8-bit): 12.0, p-value: 0.15625 The difference is NOT statistically significant.
(Original vs 4-bit): 15.0, p-value: 0.03125 The difference is statistically significant.


### friedman test just for testing purpose

In [5]:
stat, p_value = friedmanchisquare(df['Original'], df['8-bit'], df['4-bit'])
print(f"Friedman chi² statistic: {stat:.3f}, p-value: {p_value:.4f}")

pairs = [("Original", "8-bit"), ("Original", "4-bit"), ("8-bit", "4-bit")]
p_values = [ wilcoxon(df[a], df[b])[1] for a, b in pairs]
rejected, corrected_p, _, _ = multipletests(p_values, alpha=0.05, method="bonferroni")
for (a, b), p, corr_p in zip(pairs, p_values, corrected_p):
    print(f"{a} vs {b}: p={p:.4f}, corrected_p={corr_p:.4f}")

Friedman chi² statistic: 8.400, p-value: 0.0150
Original vs 8-bit: p=0.3125, corrected_p=0.9375
Original vs 4-bit: p=0.0625, corrected_p=0.1875
8-bit vs 4-bit: p=0.0625, corrected_p=0.1875


## RQ-2

In [4]:
colNames = ["ROC_AUC", "ROC_AUC.1", "ROC_AUC.2", "ROC_AUC.3"]
limit = 15
def check_roc_significance(df, score_col):
    df0 = pd.read_csv(full_url, skiprows=2, usecols=["Model", "Compression", score_col]).iloc[:limit, :]
    df1 = df0.pivot(index="Model", columns="Compression", values=score_col)
    df = df1.reset_index()
    print(df.shape)
    check_significance(df, "Original", "8-bit")
    check_significance(df, "Original", "4-bit")
    

for col in colNames:
    print(f"Checking significance for {col}")
    check_roc_significance(df, score_col=col)

colNames = ["PR_AUC", "PR_AUC.1", "PR_AUC.2", "PR_AUC.3"]
for col in colNames:
    print(f"Checking significance for {col}")
    check_roc_significance(df, score_col=col)

Checking significance for ROC_AUC
(5, 4)
(Original vs 8-bit): 15.0, p-value: 0.03125 The difference is statistically significant.
(Original vs 4-bit): 15.0, p-value: 0.03125 The difference is statistically significant.
Checking significance for ROC_AUC.1
(5, 4)
(Original vs 8-bit): 15.0, p-value: 0.03125 The difference is statistically significant.
(Original vs 4-bit): 15.0, p-value: 0.03125 The difference is statistically significant.
Checking significance for ROC_AUC.2
(5, 4)
(Original vs 8-bit): 15.0, p-value: 0.03125 The difference is statistically significant.
(Original vs 4-bit): 15.0, p-value: 0.03125 The difference is statistically significant.
Checking significance for ROC_AUC.3
(5, 4)
(Original vs 8-bit): 10.0, p-value: 0.3125 The difference is NOT statistically significant.
(Original vs 4-bit): 15.0, p-value: 0.03125 The difference is statistically significant.
Checking significance for PR_AUC
(5, 4)
(Original vs 8-bit): 10.0, p-value: 0.033944577430914495 The difference is 

/Users/nazmul/Library/Python/3.9/lib/python/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


(5, 4)
(Original vs 8-bit): 14.0, p-value: 0.0625 The difference is NOT statistically significant.
(Original vs 4-bit): 15.0, p-value: 0.03125 The difference is statistically significant.
Checking significance for PR_AUC.2
(5, 4)
(Original vs 8-bit): 9.0, p-value: 0.07206351740800766 The difference is NOT statistically significant.
(Original vs 4-bit): 15.0, p-value: 0.03125 The difference is statistically significant.
Checking significance for PR_AUC.3


/Users/nazmul/Library/Python/3.9/lib/python/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


(5, 4)
(Original vs 8-bit): 8.0, p-value: 0.13666083914614907 The difference is NOT statistically significant.
(Original vs 4-bit): 15.0, p-value: 0.03125 The difference is statistically significant.


/Users/nazmul/Library/Python/3.9/lib/python/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


## RQ-4

### RQ-4.1: Task Performance

In [7]:
limit = 39
df0 = pd.read_csv(full_url, skiprows=2, usecols=["Model", "Compression", "BLEU"]).iloc[:limit, :]
# print(df0.shape)
df1 = df0.pivot(index="Model", columns="Compression", values="BLEU")
df = df1.reset_index()
print(df.shape)
check_significance(df, "Original", "8-bit")
check_significance(df, "Original", "4-bit")

(11, 4)
(Original vs 8-bit): 44.5, p-value: 0.1826171875 The difference is NOT statistically significant.
(Original vs 4-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.


### RQ-4.2: Privacy Effectiveness

In [9]:
colNames = ["ROC_AUC", "ROC_AUC.1", "ROC_AUC.2", "ROC_AUC.3"]
limit = 39
def check_roc_significance(df, score_col):
    df0 = pd.read_csv(full_url, skiprows=2, usecols=["Model", "Compression", score_col]).iloc[:limit, :]
    df1 = df0.pivot(index="Model", columns="Compression", values=score_col)
    df = df1.reset_index()
    print(df.shape)
    check_significance(df, "Original", "8-bit")
    check_significance(df, "Original", "4-bit")
    

for col in colNames:
    print(f"Checking significance for {col}")
    check_roc_significance(df, score_col=col)

colNames = ["PR_AUC", "PR_AUC.1", "PR_AUC.2", "PR_AUC.3"]
for col in colNames:
    print(f"Checking significance for {col}")
    check_roc_significance(df, score_col=col)

Checking significance for ROC_AUC
(11, 4)
(Original vs 8-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.
(Original vs 4-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.
Checking significance for ROC_AUC.1
(11, 4)
(Original vs 8-bit): 55.0, p-value: 0.0025167541003031247 The difference is statistically significant.
(Original vs 4-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.
Checking significance for ROC_AUC.2
(11, 4)
(Original vs 8-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.
(Original vs 4-bit): 66.0, p-value: 0.00048828125 The difference is statistically significant.
Checking significance for ROC_AUC.3
(11, 4)
(Original vs 8-bit): 34.0, p-value: 0.2532590889119486 The difference is NOT statistically significant.
(Original vs 4-bit): 64.0, p-value: 0.00146484375 The difference is statistically significant.
Checking significance for PR_AUC
(11, 4)
(Origina